In [1]:


import os
import json
import pandas as pd
from datetime import datetime, timezone

import duckdb
DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"
con = duckdb.connect(DB_PATH) 
print("Connected:", DB_PATH)

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️ SHAP not installed — using permutation importance fallback.")


# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb


In [4]:

# con.close()
# print("DuckDB connection closed.")

### Training

In [35]:
train_df = con.execute('select* from prediction_data').df()
train_df = train_df[train_df["season_id"] != 20252026]
train_df = train_df.sort_values(by='game_id').reset_index(drop=True)
train_df.head()

,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct
0,20092010,2009020001,2009-10-01,15,WSH,Washington Capitals,<NA>,<NA>,0,NaN,NaN,0.466555,4,4.0,205.0,2,2.0,2,2.0,0.50000,0.500000,0,0.0,5,5.0,1.000000,1.00000,0,0.0,505.0,0,0.0,0,0.0,4.0,2.0,50.0,2,0.0,5.0,0.0,1,0.95000,1.00000,8,4,4,8,4,4,8.0,4.0,4.0,56,50,6,38,32,6,6,1,56.0,50.0,6.0,16,34,4,0,6,10,24,6,10,24,0.5,NaN,0.5,NaN,0.5,NaN,16,34,4,0,16.0,0.0,6,6,10,10,24,24,34,6,10,18,2,2,0,11.764706,20,6,6,9,95.000000,106.764706,4,6,10,18,2,2,0,1,6,6,9,0.117647,0.950000,106.764706,56,50,38,32,6,6.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,4.0,3,3.0,W,1.0,1,1,0,0,2,4,1,3,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,4,1,3,Southeast,1,5,1,1,Eastern,1,14,1,1,2,29,1,2,0,<NA>,4,1,0,0,0.0,0.466555,0.950000,12,5.00000
1,20092010,2009020001,2009-10-01,6,BOS,Boston Bruins,<NA>,<NA>,0,NaN,NaN,0.476074,5,5.0,505.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,4,4.0,0.500000,0.50000,-2,-2.0,205.0,2,2.0,0,0.0,5.0,0.0,0.0,5,2.0,4.0,50.0,4,0.88235,4.00000,3,7,-4,3,7,-4,3.0,7.0,-4.0,47,50,-3,29,32,-3,-3,1,47.0,50.0,-3.0,15,20,1,0,6,10,19,6,10,19,0.5,NaN,0.5,NaN,0.5,NaN,15,20,1,0,15.0,0.0,6,6,10,10,19,19,20,6,6,9,1,0,0,5.000000,34,6,10,18,88.235294,93.235294,1,6,6,9,1,0,0,4,6,10,18,0.050000,0.882353,93.235294,47,50,29,32,-3,-3.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,1.0,-3,-3.0,L,1.0,1,0,1,0,0,1,4,-3,0,0,1,0,1,0,0,0,0,1,4,-3,0,0,0,0,0,0,0,0,0,0,Northeast,5,5,2,5,Eastern,15,15,5,15,29,30,5,29,0,<NA>,1,4,0,0,0.0,0.000000,3.529400,12,2.00000
2,20092010,2009020002,2009-10-01,10,TOR,Toronto Maple Leafs,<NA>,<NA>,0,NaN,NaN,0.505584,3,3.0,220.0,2,2.0,2,2.0,0.66666,0.666666,0,0.0,4,4.0,0.500000,0.50000,-2,-2.0,280.0,2,2.0,0,0.0,3.0,2.0,66.7,1,2.0,4.0,50.0,4,0.85185,3.72093,15,10,5,15,10,5,15.0,10.0,5.0,92,78,14,64,58,6,14,1,92.0,78.0,14.0,29,46,3,0,12,18,38,12,18,38,0.5,NaN,0.5,N

In [36]:
# -------------------------------------------------
# Normalize made_playoffs: once 1 appears → always 1
# -------------------------------------------------
final_outcomes = (
    train_df
    .groupby(["season_id", "tri_code"])["made_playoffs"]
    .apply(lambda s: 1 if (s == 1).any() else 0)
    .reset_index()
)

train_df = (
    train_df
    .drop(columns=["made_playoffs"])
    .merge(final_outcomes, on=["season_id", "tri_code"], how="left")
)

# -------------------------------------------------
# Assertion: constant per team-season
# -------------------------------------------------
assert (
    train_df
    .groupby(["season_id", "tri_code"])["made_playoffs"]
    .nunique()
    .le(1)
    .all()
), "❌ made_playoffs varies within a team-season"


train_df = train_df.sort_values(by=['season_id','tri_code', 'game_id'])

conference_sizes = (
    train_df.groupby(["season_id", "conference"])["tri_code"]
    .nunique()
    .reset_index()
    .rename(columns={"tri_code": "conference_team_count"})
)

train_df = train_df.merge(conference_sizes, on=["season_id", "conference"], how="left")

train_df["conf_playoff_chance_baseline"] = 8 / train_df["conference_team_count"]

train_df["streak_direction"] = train_df["streak"].map({
    "W": 1,
    "L": -1
}).fillna(0).astype(int)


train_df["streak_signed"] = (
    train_df["streak_direction"] *
    train_df["streak_count"].fillna(0).astype(int)
)


train_df["streak_signed"] = train_df["streak_signed"].clip(-7, 7)


train_df.drop(columns=["streak", "streak_direction"], inplace=True)


assert (
    train_df
    .groupby(["season_id", "tri_code", "game_date"])["streak_signed"]
    .nunique()
    .le(1)
    .all()
), "❌ streak_signed varies within a team-day"

print("✅ streak_signed feature created successfully")

def bucket_rank(rank):
    if rank <= 3:
        return "tier1_elite"
    elif rank <= 6:
        return "tier2_strong"
    elif rank <= 10:
        return "tier3_bubble"
    else:
        return "tier4_bottom"
        
train_df["rank_bucket"] = train_df["conference_rank"].apply(bucket_rank)

# =====================================================
# DROP HIGH-NOISE / HIGH-RISK FEATURES (MODEL ONLY)
# =====================================================

DROP_COLS = [

    # -------------------------------------------------
    # 1. SINGLE-GAME RAW STATS (NOISY)
    # -------------------------------------------------
    "shots_for",
    "shots_against",
    "goals_for",
    "goals_against",

    "hd_sog",
    "md_sog",
    "ld_sog",

    "hd_goals",
    "md_goals",
    "ld_goals",

    "hd_against",
    "md_against",
    "ld_against",

    "game_savePct",
    "game_GAA",

    # -------------------------------------------------
    # 2. RAW PER-GAME SPECIAL TEAMS COUNTS
    # (keep cumulative / per-game rates instead)
    # -------------------------------------------------
    "pp_gf",
    "pp_ga",
    "sh_gf",
    "sh_ga",
    "times_sh",

    # -------------------------------------------------
    # 3. REDUNDANT / DOMINANT RANK FEATURES
    # (keep rank_bucket + conference_rank only)
    # -------------------------------------------------

    "division_home_rank",
    "division_road_rank",
    "division_last_10_rank",

    "conference_home_rank",
    "conference_road_rank",
    "conference_last_10_rank",

    "league_rank",
    "league_home_rank",
    "league_road_rank",
    "league_last_10_rank",



    # -------------------------------------------------
    # 4. OPTIONAL: STRONG LATE-SEASON PROXIES
    # (uncomment if probabilities converge too fast)
    # -------------------------------------------------
    # "in_playoff_spot",
]

# =====================================================
# APPLY DROP (SAFE — ignores missing)
# =====================================================

print(f"Dropping {len(DROP_COLS)} high-risk features...")

train_df_model = train_df.drop(columns=DROP_COLS, errors="ignore")

print("Remaining feature count:", train_df_model.shape[1])

train_df_model.head()



✅ streak_signed feature created successfully
Dropping 30 high-risk features...
Remaining feature count: 173


,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga_pg,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga_pg,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shot_pct,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,conference,conference_rank,wildcard_rank,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct,made_playoffs,conference_team_count,conf_playoff_chance_baseline,streak_signed,rank_bucket
0,20092010,2009020022,2009-10-03,24,ANA,Anaheim Ducks,<NA>,<NA>,0,NaN,0.000000,0.509235,7,7.0,607.0,1.0,0,0.0,0.14285,0.00,1.0,6.0,0.833333,0.83333,-1,-1.0,560.0,1.0,0.0,7.0,0.0,0.0,14,1.0,6.0,16.7,4,3,2,1,3,2,1,3.000000,2.000000,1.0,50,56,-6,34,43,-9,-6,1,50.000000,56.000000,-6.000000,19,25,1,0,5,14,18,5,14,18,0.5,NaN,0.5,NaN,0.5,NaN,19,25,1,0,19.0,0.0,5,5,14,14,18,18,4.000000,89.189189,93.189189,1,5,8,12,0,0,1,4,15,9,13,0.040000,0.891892,93.189189,50,56,34,43,-9,-9.000000,81,0,0.000000,1,0,0,0.000000,0,0,0,0,0.000,0,0,1.000000,-3,-3.000000,1.0,1,0,1,0,0,1,4,-3,0,0,1,0,1,0,0,0,0,1,4,-3,0,0,0,0,0,0,0,0,0,0,Pacific,4,Western,12,0,0,0,0.00,0.000000,3.569540,-3,4.99998,0,15.0,0.533333,-1,tier4_bottom
1,20092010,2009020029,2009-10-06,24,ANA,Anaheim Ducks,<NA>,0,1,0.000000,0.000000,0.515522,2,2.0,219.0,1.0,1,1.0,0.50000,0.50,0.0,5.0,0.400000,0.40000,-3,-3.0,452.0,3.0,0.0,9.0,1.0,11.1,14,4.0,11.0,36.4,12,7,7,0,10,9,1,5.000000,4.500000,0.5,35,38,-3,26,28,-2,-9,2,42.500000,47.000000,-4.500000,12,19,3,0,4,8,15,4,8,15,0.5,NaN,0.5,NaN,0.5,NaN,31,44,4,0,15.5,0.0,9,9,22,22,33,33,15.789474,87.500000,103.289474,4,7,15,22,1,2,1,8,23,15,32,0.090909,0.884058,97.496706,85,94,60,71,-11,-5.500000,80,0,0.000000,1,1,0,0.000000,0,0,0,1,0.250,41,0,2.000000,-4,-2.000000,1.0,2,0,1,1,1,4,8,-4,0,0,1,0,1,0,0,0,0,1,4,-3,1,0,0,1,0,0,1,3,4,-1,Pacific,5,Western,13,0,0,1,0.50,0.128881,3.326286,0,2.00000,0,15.0,0.533333,0,tier4_bottom
2,20092010,2009020035,2009-10-08,24,ANA,Anaheim Ducks,<NA>,0,2,0.000000,0.166667,0.521966,2,2.0,142.0,2.0,2,2.0,1.00000,1.00,0.0,6.0,1.166666,1.00000,1,1.0,719.0,0.0,1.0,11.0,3.0,27.3,8,4.0,17.0,23.5,10,1,2,-1,11,11,0,3.666667,3.666667,0.0,49,57,-8,37,40,-3,-17,3,44.666667,50.333333,-5.666667,23,30,6,0,7,17,17,7,17,17,0.5,NaN,0.5,NaN,0.5,NaN,54,74,10,0,18.0,0.0,16,16,39,39,50,50,20.000000,97.058824,117.058824,10,13,30,32,3,6,1,9,25,24,55,0.135135,0.912621,104.775649,134,151,97,111,-14,-4.666667,79,1,0.333333,1,1,1,0.333333,1,0,0,3,0.500,82,0,3.333333,1,0.333333,1.0,3,1,1,1,3,10,9,1,1,1,1,0,1,0,0,0,0,1,4,-3,2,1,0,1,1,1,3,9,5,4,Pac

### Testing

In [37]:
test_df = con.execute('select* from prediction_data').df()
test_df = test_df[test_df["season_id"] == 20252026]
test_df = test_df.sort_values(by='game_id').reset_index(drop=True)
test_df.head()

,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct
0,20252026,2025020001,2025-10-07,16,CHI,Chicago Blackhawks,0,<NA>,0,NaN,NaN,0.497453,3,3.0,360.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,2,2.0,0.50,0.50,-1,-1.0,222.0,1,1.0,0,0.0,3.0,0.0,0.0,4,1.0,2.0,50.0,4,0.91891,3.04740,2,8,-6,2,8,-6,2.0,8.0,-6.0,50,49,1,33,34,-1,1,1,50.0,49.0,1.0,28,19,2,65,11,14,26,8,21,23,0.578947,0.421053,0.400000,0.600000,0.530612,0.469388,28,19,2,65,28.0,65.0,11,8,14,21,26,23,19,3,5,12,0,2,0,10.526316,37,10,11,16,91.891892,102.418208,2,3,5,12,0,2,0,3,10,11,16,0.105263,0.918919,102.418208,50,49,33,34,-1,-1.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,2.0,-1,-1.0,L,1.0,1,0,1,0,0,2,3,-1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,2,3,-1,Central,8,1,8,8,Western,15,3,16,15,30,7,32,30,9,2,3,0,0,0,0.000000,2.800286,6.0,1,NaN
1,20252026,2025020001,2025-10-07,13,FLA,Florida Panthers,1,<NA>,0,NaN,NaN,0.497126,2,2.0,222.0,1,1.0,1,1.0,0.50000,0.500000,0,0.0,3,3.0,1.00,1.00,0,0.0,360.0,0,0.0,0,0.0,2.0,1.0,50.0,1,0.0,3.0,0.0,1,0.89473,2.00000,12,6,6,12,6,6,12.0,6.0,6.0,69,69,0,53,52,1,0,1,69.0,69.0,0.0,37,37,3,65,17,24,28,20,17,31,0.459459,0.540541,0.585366,0.414634,0.474576,0.525424,37,37,3,65,37.0,65.0,17,20,24,17,28,31,37,10,11,16,2,0,1,8.108108,19,3,5,12,89.473684,97.581792,3,10,11,16,2,0,1,2,3,5,12,0.081081,0.894737,97.581792,69,69,53,52,1,1.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3.0,1,1.0,W,1.0,1,1,0,0,2,3,2,1,1,1,1,1,0,0,1,1,2,3,2,1,0,0,0,0,0,0,0,0,0,0,Atlantic,1,1,4,1,Eastern,2,1,7,2,3,1,12,3,0,3,2,0,0,0,0.497126,1.789460,6.0,3,NaN
2,20252026,2025020002,2025-10-07,3,NYR,New York Rangers,0,<NA>,0,NaN,NaN,0.513884,2,2.0,240.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,1,1.0,1.00,1.00,0,0.0,120.0,0,0.0,0,0.0,2.0,0.0,0.0,4,0.0,1.0,0.0,1,0.96551,1.00642,7,9,-2,7,9,-2,7.0,9.0,-2.0,60,60,0,36,36,0,0,1,60.0,60.0,0.0,34,25,0,66,14,19,29,16

### Feature Engineering

In [ ]:

conference_sizes = (
    test_df.groupby(["season_id", "conference"])["tri_code"]
    .nunique()
    .reset_index()
    .rename(columns={"tri_code": "conference_team_count"})
)

test_df = test_df.merge(conference_sizes, on=["season_id", "conference"], how="left")

test_df["conf_playoff_chance_baseline"] = 8 / test_df["conference_team_count"]
test_df["streak_direction"] = test_df["streak"].map({
    "W": 1,
    "L": -1
}).fillna(0).astype(int)


test_df["streak_signed"] = (
    test_df["streak_direction"] *
    test_df["streak_count"].fillna(0).astype(int)
)


test_df["streak_signed"] = test_df["streak_signed"].clip(-7, 7)


test_df.drop(columns=["streak", "streak_direction"], inplace=True)


assert (
    test_df
    .groupby(["season_id", "tri_code", "game_date"])["streak_signed"]
    .nunique()
    .le(1)
    .all()
), "❌ streak_signed varies within a team-day"

print("✅ streak_signed feature created successfully")
def bucket_rank(rank):
    if rank <= 3:
        return "tier1_elite"
    elif rank <= 6:
        return "tier2_strong"
    elif rank <= 10:
        return "tier3_bubble"
    else:
        return "tier4_bottom"
        
test_df["rank_bucket"] = test_df["conference_rank"].apply(bucket_rank)




✅ streak_signed feature created successfully
Dropping 30 high-risk features...
Remaining feature count: 173


,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga_pg,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga_pg,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shot_pct,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,conference,conference_rank,wildcard_rank,made_playoffs,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct,conference_team_count,conf_playoff_chance_baseline,streak_signed,rank_bucket
0,20252026,2025020001,2025-10-07,16,CHI,Chicago Blackhawks,0,<NA>,0,NaN,NaN,0.497453,3,3.0,360.0,0.0,0,0.0,0.00000,0.000000,0.0,2.0,0.50,0.50,-1,-1.0,222.0,1.0,0.0,3.0,0.0,0.0,4,1.0,2.0,50.0,4,2,8,-6,2,8,-6,2.0,8.0,-6.0,50,49,1,33,34,-1,1,1,50.0,49.0,1.0,28,19,2,65,11,14,26,8,21,23,0.578947,0.421053,0.400000,0.600000,0.530612,0.469388,28,19,2,65,28.0,65.0,11,8,14,21,26,23,10.526316,91.891892,102.418208,2,3,5,12,0,2,0,3,10,11,16,0.105263,0.918919,102.418208,50,49,33,34,-1,-1.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,2.0,-1,-1.0,1.0,1,0,1,0,0,2,3,-1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,2,3,-1,Central,8,Western,15,9,2,0,0,0.000000,2.800286,6.0,1,NaN,16,0.5,-1,tier4_bottom
1,20252026,2025020001,2025-10-07,13,FLA,Florida Panthers,1,<NA>,0,NaN,NaN,0.497126,2,2.0,222.0,1.0,1,1.0,0.50000,0.500000,0.0,3.0,1.00,1.00,0,0.0,360.0,0.0,0.0,2.0,1.0,50.0,1,0.0,3.0,0.0,1,12,6,6,12,6,6,12.0,6.0,6.0,69,69,0,53,52,1,0,1,69.0,69.0,0.0,37,37,3,65,17,24,28,20,17,31,0.459459,0.540541,0.585366,0.414634,0.474576,0.525424,37,37,3,65,37.0,65.0,17,20,24,17,28,31,8.108108,89.473684,97.581792,3,10,11,16,2,0,1,2,3,5,12,0.081081,0.894737,97.581792,69,69,53,52,1,1.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3.0,1,1.0,1.0,1,1,0,0,2,3,2,1,1,1,1,1,0,0,1,1,2,3,2,1,0,0,0,0,0,0,0,0,0,0,Atlantic,1,Eastern,2,0,3,0,0,0.497126,1.789460,6.0,3,NaN,16,0.5,1,tier1_elite
2,20252026,2025020002,2025-10-07,3,NYR,New York Rangers,0,<NA>,0,NaN,NaN,0.513884,2,2.0,240.0,0.0,0,0.0,0.00000,0.000000,0.0,1.0,1.00,1.00,0,0.0,120.0,0.0,0.0,2.0,0.0,0.0,4,0.0,1.0,0.0,1,7,9,-2,7,9,-2,7.0,9.0,-2.0,60,60,0,36,36,0,0,1,60.0,60.0,0.0,34,25,0,66,14,19,29,16,19,24,0.466667,0.533333,0.500000,0.500000,0.547170,0.452830,34,25,0,66,34.0,66.0,14,16,19,19,29,24,0.000000,90.322581,90.322581,0,4,8,14,0,0,0,3,10,7,14,0.000000,0.903226,90.322581,60,60,36,36,0,0.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,0.0,-3,-3.0,1.0,1,0,1,0,0,0,3,-3,0,0,1,0,1,0,0,0,0,0,3,-3,0,0,0,0,0,0,0,0,0,0,Metropolitan,8,Eastern,16,10,0,0,0,0.000000,0.971709,6.0,1,NaN,16,0.5,-1,tier4_bottom
3,20252026,2025020002,2025-10-07,5,PIT,Pittsburgh Pengu

In [ ]:

# =====================================================
# DROP HIGH-NOISE / HIGH-RISK FEATURES (MODEL ONLY)
# =====================================================

DROP_COLS = [

    # -------------------------------------------------
    # 1. SINGLE-GAME RAW STATS (NOISY)
    # -------------------------------------------------
    "shots_for",
    "shots_against",
    "goals_for",
    "goals_against",

    "hd_sog",
    "md_sog",
    "ld_sog",

    "hd_goals",
    "md_goals",
    "ld_goals",

    "hd_against",
    "md_against",
    "ld_against",

    "game_savePct",
    "game_GAA",

    # -------------------------------------------------
    # 2. RAW PER-GAME SPECIAL TEAMS COUNTS
    # (keep cumulative / per-game rates instead)
    # -------------------------------------------------
    "pp_gf",
    "pp_ga",
    "sh_gf",
    "sh_ga",
    "times_sh",

    # -------------------------------------------------
    # 3. REDUNDANT / DOMINANT RANK FEATURES
    # (keep rank_bucket + conference_rank only)
    # -------------------------------------------------

    "division_home_rank",
    "division_road_rank",
    "division_last_10_rank",

    "conference_home_rank",
    "conference_road_rank",
    "conference_last_10_rank",

    "league_rank",
    "league_home_rank",
    "league_road_rank",
    "league_last_10_rank",



    # -------------------------------------------------
    # 4. OPTIONAL: STRONG LATE-SEASON PROXIES
    # (uncomment if probabilities converge too fast)
    # -------------------------------------------------
    # "in_playoff_spot",
]

# =====================================================
# APPLY DROP (SAFE — ignores missing)
# =====================================================

print(f"Dropping {len(DROP_COLS)} high-risk features...")

current_df = test_df.drop(columns=DROP_COLS, errors="ignore")

print("Remaining feature count:", current_df.shape[1])

current_df.head()


In [42]:
current_df["games_played"].describe()


count    1443.000000
mean        9.933472
std         6.238199
min         1.000000
25%         4.000000
50%         9.000000
75%        15.000000
max        24.000000
Name: games_played, dtype: float64

In [ ]:

current_df[["tri_code", "game_date", "games_played"]].head(10)


### Loading

In [ ]:
# =====================================================
# LOAD MODEL ARTIFACTS
# =====================================================

from prediction_model import load_and_score
from joblib import load

bundle = load("models/playoff_models_wc_field_pressure_v4.joblib")

pipeline = bundle["pipeline"]
FEATURE_COLS = bundle["features"]

xgb_model = pipeline.named_steps["model"]
scaler = pipeline.named_steps["scaler"]

# =====================================================
# 1. SCORE CURRENT DATA (adds WC features)
# =====================================================

current_df = load_and_score(current_df)
# =====================================================
# PRESENTATION LAYER (NON-MODEL)
# =====================================================

def season_bucket(gp):
    if gp <= 20:
        return "early"
    elif gp <= 50:
        return "mid"
    else:
        return "late"

def adjust_for_uncertainty(p, bucket):
    """
    Compress early-season probabilities toward 50%.
    Presentation-only — does NOT affect model.
    """
    if bucket == "early":
        return 0.45 * p + 0.275
    return p

# Ensure season context exists
if "season_bucket" not in current_df.columns:
    current_df["season_bucket"] = (
        current_df["games_played"].apply(season_bucket)
    )

# Display-safe probability
current_df["playoff_prob_display"] = current_df.apply(
    lambda r: adjust_for_uncertainty(
        r["playoff_prob_row"],
        r["season_bucket"]
    ),
    axis=1
)


# =====================================================
# 2. BUILD SHAP FEATURE MATRIX (AFTER scoring)
# =====================================================

LEAK_COLS = [
    "game_id", "game_date", "date",
    "clinch_type", "made_playoffs",
    "team_name", "opponent",
    "season_id"
]

ID_COLS = ["team_id", "opponent_id", "tri_code"]

X_shap = (
    current_df
    .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
    .select_dtypes(include="number")
    .fillna(0)
    .reindex(columns=FEATURE_COLS, fill_value=0)
)

# =====================================================
# 3. APPLY SCALER (what the model actually sees)
# =====================================================

X_shap_scaled = scaler.transform(X_shap)

X_shap_scaled = pd.DataFrame(
    X_shap_scaled,
    columns=X_shap.columns,
    index=X_shap.index
)


ModuleNotFoundError: No module named 'prediction_model'

In [ ]:

current_df.head()


In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap_scaled)

def explain_team(
    team_code: str,
    current_df: pd.DataFrame,
    X_shap_scaled: pd.DataFrame,
    shap_values,
    explainer,
    feature_cols,
    n=8
):
    """
    Explain why a team is at its current playoff odds.
    Returns helpful + harmful driver tables.
    """

    # -------------------------------------------------
    # Locate latest row for team
    # -------------------------------------------------
    team_mask = current_df["tri_code"] == team_code
    if not team_mask.any():
        raise ValueError(f"Team {team_code} not found")

    row_idx = current_df.loc[team_mask].index[-1]

    # -------------------------------------------------
    # Extract SHAP row
    # -------------------------------------------------
    shap_row = pd.Series(
        shap_values[row_idx],
        index=feature_cols
    ).sort_values()

    most_harmful = shap_row.head(n)
    most_helpful = shap_row.tail(n)

    # -------------------------------------------------
    # Season stage (defensive)
    # -------------------------------------------------
    if "season_bucket" in current_df.columns:
        season_stage = current_df.loc[row_idx, "season_bucket"]
    elif "games_played" in current_df.columns:
        gp = current_df.loc[row_idx, "games_played"]
        season_stage = (
            "early" if gp <= 20 else
            "mid" if gp <= 50 else
            "late"
        )
    else:
        season_stage = "unknown"

    playoff_prob = current_df.loc[row_idx, "playoff_prob_display"]

    # -------------------------------------------------
    # Print narrative header
    # -------------------------------------------------
    print(
        f"\n📊 {team_code} PLAYOFF EXPLANATION "
        f"({season_stage.upper()} SEASON)\n"
        f"Displayed Odds: {playoff_prob:.1%}\n"
    )

    print("🔻 Most Harmful Drivers")
    display(most_harmful.round(3))

    print("\n🔺 Most Helpful Drivers")
    display(most_helpful.round(3))

    return most_helpful, most_harmful


### Team Exploration

In [ ]:


explain_team(
    team_code="BOS",
    current_df=current_df,
    X_shap_scaled=X_shap_scaled,
    shap_values=shap_values,
    explainer=explainer,
    feature_cols=FEATURE_COLS,
    n=8
)


In [ ]:
row_idx = current_df.index[current_df["tri_code"] == "BOS"][-1]

shap.force_plot(
    explainer.expected_value,
    shap_values[row_idx],
    X_shap_scaled.loc[[row_idx]],
    matplotlib=True
)


In [ ]:
shap_values = explainer.shap_values(X_shap_scaled)


In [ ]:
shap_df = pd.DataFrame(
    shap_values,
    columns=FEATURE_COLS,
    index=current_df.index
)


In [ ]:
latest_idx = (
    current_df
    .sort_values(["team_id", "game_date"])
    .groupby("team_id")
    .tail(1)
    .index
)

shap_latest = shap_df.loc[latest_idx]
meta_latest = current_df.loc[latest_idx]


In [ ]:
def top_negative_drivers(shap_row, n=3):
    sorted_row = shap_row.sort_values()
    return list(zip(sorted_row.index[:n], sorted_row.values[:n]))


In [ ]:
records = []

for idx, shap_row in shap_latest.iterrows():
    sinks = top_negative_drivers(shap_row, n=3)

    records.append({
        "team_name": meta_latest.loc[idx, "team_name"],
        "tri_code": meta_latest.loc[idx, "tri_code"],
        "season_id": meta_latest.loc[idx, "season_id"],
        "biggest_sinker": f"{sinks[0][0]} ({sinks[0][1]:.3f})",
        "second_biggest_sinker": f"{sinks[1][0]} ({sinks[1][1]:.3f})",
        "third_biggest_sinker": f"{sinks[2][0]} ({sinks[2][1]:.3f})",
        "biggest_sinker_value": sinks[0][1],
    })


In [ ]:
sinker_df = pd.DataFrame(records)

sinker_df = sinker_df.sort_values(
    "team_name"
).reset_index(drop=True)

sinker_df


## STEP 1 — Tag features by pressure sensitivity

In [ ]:
# ----------------------------------------
# Feature interpretation taxonomy
# v1.1 — pressure-refined
# ----------------------------------------

PRESSURE_SENSITIVE = {
    # -------------------------------------------------
    # Special teams & discipline (amplify race pressure)
    # -------------------------------------------------
    "pk_goals_against_pg",
    "pk_pct",
    "pk_time_on_ice_pg",
    "penalties_taken_pg",
    "c_pp_net_g",
    "times_sh_pg",

    # -------------------------------------------------
    # Transition / rush (harder to sustain under pressure)
    # -------------------------------------------------
    "rush_for_per_game",
    "rush_diff_per_game",
    "rush_against_per_game",
    "rush_shots_against",

    # -------------------------------------------------
    # Margins & close-game resolution
    # -------------------------------------------------
    "goal_diff",
    "home_goal_diff",
    "goal_diff_pctg",
    "one_goal_win_pct",
    "rotw_pctg",
    "rw",
    "regulation_wins",
    "ot_wins",
    "ot_losses",
    "so_wins",
    "so_losses",

    # -------------------------------------------------
    # Standings conversion & playoff leverage
    # (internal performance under external pressure)
    # -------------------------------------------------
    "points",
    "points_pct",
    "projected_points",
    "win_pct_before",

    # -------------------------------------------------
    # Possession balance under stress
    # -------------------------------------------------
    "c_shot_pct",
    "corsi_diff_per_game",
    "c_corsi_diff",
    "c_pdo",

    # -------------------------------------------------
    # Road resilience (schedule + mental stress test)
    # -------------------------------------------------
    "road_goal_diff",
    "road_rw",
    "road_ot_losses",
    "road_points",
    "road_wins",
    "road_rotw",

    # -------------------------------------------------
    # Short-horizon performance (signal compression zone)
    # -------------------------------------------------
    "last_10_rw",
    "last_10_goal_diff",
}

PRESSURE_AGNOSTIC = {
    # -------------------------------------------------
    # Possession & shot volume (baseline quality)
    # -------------------------------------------------
    "corsi_for_pg",
    "SCF_pg",
    "xGF_pg",
    "shots_for_pg",

    # -------------------------------------------------
    # Offensive ceiling (less race-dependent)
    # -------------------------------------------------
    "goals_for_pg",
    "pp_goals_pg",

    # -------------------------------------------------
    # Goaltending baseline (talent > context)
    # -------------------------------------------------
    "c_sv_pct",
}

EXOGENOUS_PRESSURE = {
    # -------------------------------------------------
    # Structural / environment features (not controllable)
    # -------------------------------------------------
    "division_rank",
    "conference_rank",
    "wildcard_rank",
    "conference_team_count",

    # -------------------------------------------------
    # Wild-card race geometry
    # -------------------------------------------------
    "wc_points_margin",
    "wc_points_pct_margin",
    "wc_teams_within_2pts",
    "wc_teams_within_4pts",
    "wc_teams_within_6pts",

    # -------------------------------------------------
    # Schedule / opponent difficulty
    # -------------------------------------------------
    "sos_future",
}

CONTEXT_AMPLIFIED_EXECUTION = {
    # Core performance that matters more in tight races
    "points_pct",
    "goal_diff_pctg",
    "projected_points",
    "pt_pct_x_sos",

    # Possession & shot quality under pressure
    "corsi_diff_per_game",
    "c_corsi_diff",
    "c_shot_pct",

    # Transition margins
    "rush_for_per_game",
    "rush_diff_per_game",

    # Home / road split effects
    "road_goal_diff",
    "road_points",
    "home_goal_diff",

    # Schedule-scaled metrics
    "conference_team_count",
}



In [ ]:
def classify_feature(feature):
    if feature in EXOGENOUS_PRESSURE:
        return "Environment"
    elif feature in PRESSURE_SENSITIVE:
        return "Pressure-Sensitive"
    elif feature in PRESSURE_AGNOSTIC:
        return "Pressure-Agnostic"
    elif feature in CONTEXT_AMPLIFIED_EXECUTION:
        return "Context-Amplified"
    else:
        return "Unclassified"


## STEP 2 — Build a team robustness diagnostic

In [ ]:
def team_robustness_breakdown(
    team_code,
    current_df,
    shap_df,
):
    """
    Quantifies how much a team's negative SHAP impact
    comes from environment vs pressure-sensitive performance.
    """

    # Locate latest snapshot
    team_rows = current_df[current_df["tri_code"] == team_code]
    if team_rows.empty:
        raise ValueError(f"{team_code} not found")

    row_idx = team_rows.sort_values("game_date").index[-1]
    shap_row = shap_df.loc[row_idx]

    # Only negative contributors
    negative = shap_row[shap_row < 0]

    records = []
    for feature, value in negative.items():
        records.append({
            "feature": feature,
            "impact": value,
            "category": classify_feature(feature)
        })

    df = pd.DataFrame(records)

    summary = (
        df.groupby("category")["impact"]
        .sum()
        .sort_values()
    )

    return df.sort_values("impact"), summary


In [ ]:
detail, summary = team_robustness_breakdown(
    team_code="BOS",
    current_df=current_df,
    shap_df=shap_df
)

print("📉 Negative SHAP impact by category:")
display(summary)

print("\n🔍 Biggest individual sinkers:")
display(detail.head(10))


In [ ]:
detail, summary = team_robustness_breakdown(
    "BOS",
    current_df,
    shap_df
)

detail.query("category == 'Unclassified'")


## Pressure Leverage Table

In [ ]:
FEATURE_METADATA = {

    # =====================================================
    # PRESSURE-SENSITIVE (FIXABLE EXECUTION)
    # =====================================================
    # These represent true execution under stress:
    # discipline, special teams, rush defense, close games

    # Transition & rush defense
    "rush_against_per_game":    ("Pressure-Sensitive", "High"),

    # Close-game & resolution skills
    "ot_wins":                  ("Pressure-Sensitive", "Low"),
    "rw":                       ("Pressure-Sensitive", "Medium"),
    "home_rotw":                ("Pressure-Sensitive", "Medium"),
    "last_10_rw":               ("Pressure-Sensitive", "Medium"),

    # =====================================================
    # CONTEXT-AMPLIFIED EXECUTION (MARGINS MATTER MORE)
    # =====================================================
    # These are NOT failures under pressure —
    # they are normal performance metrics whose impact
    # increases when the race is tight

    # Core performance / margins
    "points_pct":               ("Context-Amplified", "Low"),
    "gf_pctg":                  ("Context-Amplified", "Low"),
    "goal_diff_pctg":           ("Context-Amplified", "Low"),
    "projected_points":         ("Context-Amplified", "Low"),
    "pt_pct_x_sos":             ("Context-Amplified", "Low"),

    # Possession & shot quality
    "c_shot_pct":               ("Context-Amplified", "Low"),
    "corsi_diff_per_game":      ("Context-Amplified", "Low"),
    "c_corsi_diff":             ("Context-Amplified", "Low"),
    "c_pdo":                    ("Context-Amplified", "Low"),

    # Transition margins (offensive side)
    "rush_for_per_game":        ("Context-Amplified", "Low"),
    "rush_diff_per_game":       ("Context-Amplified", "Low"),

    # Home / road splits
    "road_goal_diff":           ("Context-Amplified", "Low"),
    "road_points":              ("Context-Amplified", "Low"),
    "road_rw":                  ("Context-Amplified", "Low"),
    "road_ot_losses":           ("Context-Amplified", "Low"),
    "home_goal_diff":           ("Context-Amplified", "Low"),

    # =====================================================
    # PRESSURE-AGNOSTIC (BASELINE QUALITY)
    # =====================================================
    # These describe team ceiling, not race behavior

    "SCF_pg":                   ("Pressure-Agnostic", "Low"),
    "shots_for_pg":             ("Pressure-Agnostic", "Low"),
    "xGF_pg":                   ("Pressure-Agnostic", "Low"),
    "c_sv_pct":                 ("Pressure-Agnostic", "Low"),

    # =====================================================
    # ENVIRONMENT (NOT FIXABLE)
    # =====================================================
    # Structural constraints of the league / standings

    "division_rank":            ("Environment", "None"),
    "conference_rank":          ("Environment", "None"),
    "wildcard_rank":            ("Environment", "None"),
    "wc_points_margin":         ("Environment", "None"),
    "wc_teams_within_2pts":     ("Environment", "None"),
    "wc_teams_within_4pts":     ("Environment", "None"),
    "wc_teams_within_6pts":     ("Environment", "None"),
    "conference_team_count":    ("Environment", "None"),
    "sos_future":               ("Environment", "None"),
}


In [ ]:
def build_pressure_leverage_table(
    shap_series: pd.Series,
    feature_metadata: dict,
    n=15
):
    df = (
        shap_series
        .sort_values()
        .head(n)
        .reset_index()
        .rename(columns={"index": "feature", 0: "shap_impact"})
    )

    df[["category", "fixability"]] = df["feature"].apply(
        lambda f: feature_metadata.get(f, ("Unclassified", "Unknown"))
    ).apply(pd.Series)

    return df


How to read it

High fixability + large negative SHAP = coaching / systems leverage

Medium fixability = tactical emphasis

Environment = context, not blame

This table answers:

“If XYZ team improves just two things, what should they be?”

In [ ]:
leverage = build_pressure_leverage_table(shap_row, FEATURE_METADATA)
leverage


## Pressure Sensitivity Score

“How fragile is this team in a tight race?”

This becomes a single number you can track over time. Pressure Sensitivity Score =
(sum of negative Pressure-Sensitive SHAP) ÷ (total negative SHAP)


In [ ]:
def pressure_sensitivity_score(shap_series, feature_metadata):
    neg = shap_series[shap_series < 0]

    total_neg = neg.sum()
    if total_neg == 0:
        return 0.0

    ps_neg = neg[
        neg.index.map(
            lambda f: feature_metadata.get(f, ("", ""))[0]
        ) == "Pressure-Sensitive"
    ].sum()

    return float(ps_neg / total_neg)


Interpretation guide:
Score	Meaning
- < 0.25	Stable contender
- 0.25–0.45	Normal playoff pressure
- 0.45–0.65	Bubble-fragile
- "> 0.65"	Execution-dependent team

Boston being high here = race amplifies weaknesses

In [ ]:
ps_score = pressure_sensitivity_score(shap_row, FEATURE_METADATA)
ps_score


## Environment vs Execution Visual

“Is this luck or performance?”

This is the chart that tells the story instantly.

In [ ]:
def shap_by_category(shap_series, feature_metadata):
    df = shap_series.reset_index()
    df.columns = ["feature", "impact"]

    df["category"] = df["feature"].apply(
        lambda f: feature_metadata.get(f, ("Unclassified",))[0]
    )

    return (
        df[df["impact"] < 0]
        .groupby("category")["impact"]
        .sum()
        .sort_values()
    )


In [ ]:
import matplotlib.pyplot as plt

cat_impact = shap_by_category(shap_row, FEATURE_METADATA)

plt.figure(figsize=(7, 4))
cat_impact.plot(kind="barh")

plt.title("Negative Playoff Pressure Drivers", fontsize=13)
plt.xlabel("Total SHAP Impact")
plt.ylabel("")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


How to read it

Pressure-Sensitive dominating → fixable execution issues

Environment dominating → schedule / race reality

Pressure-Agnostic dominating → roster ceiling issues

“This team is hurting itself more than the league is hurting it.”

≥ 0.85  → High-confidence territory
0.65–0.85 → Solid contender
0.45–0.65 → True bubble
0.25–0.45 → Outside but alive
< 0.25 → Long shot


In [ ]:
def playoff_tier(prob, pressure_sensitivity):
    if prob >= 0.85 and pressure_sensitivity <= 0.40:
        return "Lock"
    elif prob >= 0.65 and pressure_sensitivity <= 0.45:
        return "Likely"
    elif prob >= 0.45:
        return "Bubble"
    elif prob >= 0.25 and pressure_sensitivity > 0.30:
        return "Likely Out"
    else:
        return "Out"


In [ ]:
latest_snapshot = (
    current_df
    .sort_values(["team_id", "game_date"])
    .groupby("team_id", as_index=False)
    .tail(1)
)

latest_snapshot.head()

In [ ]:
latest_snapshot["playoff_tier"] = latest_snapshot.apply(
    lambda r: playoff_tier(
        r["playoff_prob_display"],
        r["pressure_sensitivity_score"]
    ),
    axis=1
)
